# 03 — 독립성 분석 · Gate 1 재판정 (M4 수정판)

**능동수익 정의 수정:** `전략 − B&H` → `전략 − ee` (동일노출 ee 차감으로 공통 주식베타 제거)

**공통기간:** 1990-01-31~ (realized·blend 워밍업 바닥, N=9148)

**★ 사전 등록 판정 임계 (결과 보기 전 고정 — 불변):**
- `|ρ_active| < 0.30` → 저상관 (기준2 포함 자격)
- `|ρ_active| > 0.50` → 고상관 (분산이득 약함)
- `0.30 ≤ |ρ| ≤ 0.50` → 회색지대, 신호 w_t 상관(i) 보조 병기

**두 CSV 병존:**
- `independence_active_bh_corr.csv`: 구 정의 (전략−B&H), 공통 주식베타 잔류
- `independence_active_corr.csv`: 새 정의 (전략−ee), Gate 1 판정 기준


In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if "" not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings, yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from src.data_loader import load_all
from src.indicators.base import standalone_data
from src.indicators.volatility import VolatilityIndicator
from src.indicators.credit import CreditIndicator
from src.indicators.trend import TrendIndicator
from src.backtest import run
from src.benchmarks import buy_and_hold, equal_exposure
from src.metrics import summary, avg_exposure

print("cwd:", os.getcwd())


In [ ]:
# ★ Gate 1 사전 등록 임계 (결과 보기 전 고정 — 사후 변경 금지)
CORR_LOW  = 0.30   # |ρ| < 0.30  → 저상관 (기준2 포함 자격)
CORR_HIGH = 0.50   # |ρ| > 0.50  → 고상관 (분산이득 약함)
# 0.30 ≤ |ρ| ≤ 0.50 → 회색지대, (i) 보조 병기

COMMON_START = pd.Timestamp("1990-01-31")

with open('config/base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open('config/indicators/credit.yaml') as f:
    credit_yaml = yaml.safe_load(f)
cfg = {
    **base_cfg,
    'percentile_window': credit_yaml['percentile_window'],
    'theta_low_pct':     credit_yaml['map']['theta_low_pct'],
    'theta_high_pct':    credit_yaml['map']['theta_high_pct'],
}
data_raw = load_all(cfg)
print(f"CORR_LOW={CORR_LOW}, CORR_HIGH={CORR_HIGH}, COMMON_START={COMMON_START.date()}")


In [ ]:
VARIANTS = {
    'vix':      (VolatilityIndicator(), {**cfg, 'vol_estimator':'vix'}, "volatility"),
    'realized': (VolatilityIndicator(), {**cfg, 'vol_estimator':'realized','realized_lookback':21}, "volatility"),
    'blend':    (VolatilityIndicator(), {**cfg, 'vol_estimator':'blend'}, "volatility"),
    'credit':   (CreditIndicator(),    cfg, "credit"),
    'ma200':    (TrendIndicator(),     {**cfg, 'rule':'ma200','w_floor':0.0}, "trend"),
    'tsmom12':  (TrendIndicator(),     {**cfg, 'rule':'tsmom12','w_floor':0.0}, "trend"),
}
FAMILIES = {'vix':'vol','realized':'vol','blend':'vol','credit':'credit','ma200':'trend','tsmom12':'trend'}
NAMES = list(VARIANTS.keys())

signals_common={}; returns_strat={}; returns_ee={}; returns_bnh={}; common_meta={}

for name,(ind,c,src) in VARIANTS.items():
    d=standalone_data(data_raw,src); w_full=ind.signal(d,c); w=w_full.loc[COMMON_START:].dropna()
    r_eq=d['sp500tr'].pct_change().reindex(w.index).fillna(0.0)
    rf=d['rf'].reindex(w.index); rf_d=rf/100/252
    res=run(w,r_eq,rf,c); mw=avg_exposure(res['weights'])
    bnh=buy_and_hold(r_eq,rf,c); ee=equal_exposure(mw,r_eq,rf,c)
    m_s=summary(res['equity_net'],res['returns_net'],r_eq,rf_d,res['weights'],res['turnover'])
    m_e=summary(ee['equity_net'],ee['returns_net'],r_eq,rf_d,ee['weights'],ee['turnover'])
    signals_common[name]=w; returns_strat[name]=res['returns_net']
    returns_ee[name]=ee['returns_net']; returns_bnh[name]=bnh['returns_net']
    common_meta[name]=dict(mean_w=mw,m_strat=m_s,m_ee=m_e)
    print(f"{name:8s}: N={len(w)}, mean_w={mw:.4f}")

common_idx=signals_common['realized'].index
for n in NAMES: common_idx=common_idx.intersection(returns_strat[n].index)
bnh_ref=returns_bnh['vix'].reindex(common_idx)
print(f"\n공통 분석 기간: {common_idx[0].date()} ~ {common_idx[-1].date()}, N={len(common_idx)}")


## 1. 공통기간 재백테스트 — common1990_metrics.csv

In [ ]:
rows=[]
for name in NAMES:
    d=common_meta[name]
    for label,m in [(f'strat_{name}',d['m_strat']),(f"ee({d['mean_w']:.3f})",d['m_ee'])]:
        rows.append({'variant':name,'family':FAMILIES[name],'strategy':label,
                     'eval_start':str(COMMON_START.date()),
                     'CAGR':f"{m['cagr']:.2%}",'Vol':f"{m['annual_vol']:.2%}",
                     'Sharpe':f"{m['sharpe']:.3f}",'Sortino':f"{m['sortino']:.3f}",
                     'MDD':f"{m['mdd']:.2%}",'Calmar':f"{m['calmar']:.3f}",
                     'UpCap':f"{m['up_capture']:.2%}",'DnCap':f"{m['down_capture']:.2%}",
                     'Turnover/yr':f"{m['annual_turnover']:.2f}",'AvgExp':f"{m['avg_exposure']:.3f}"})
df_cmn=pd.DataFrame(rows)
df_cmn.to_csv('results/common1990_metrics.csv',index=False)
print(df_cmn.to_string(index=False))


## 2. 베타 제거 진단: 전략−B&H vs 전략−ee 상관 비교

In [ ]:
# 진단: |ρ(strat-BH, BH)| vs |ρ(strat-ee, BH)|
# 후자가 전자보다 작으면 성공. 역전(후자≥전자) 시만 멈춤.
print(f"{'variant':8s}  {'|ρ(strat-BH,BH)|':>18}  {'|ρ(strat-ee,BH)|':>18}  {'낙폭':>7}  판정")
print("-"*72)
for name in NAMES:
    s=returns_strat[name].reindex(common_idx)
    e=returns_ee[name].reindex(common_idx)
    b=bnh_ref
    r_bh=abs(float((s-b).corr(b))); r_ee=abs(float((s-e).corr(b)))
    drop=r_bh-r_ee
    tag=f"성공(낙폭 {drop:+.3f})" if drop>0 else f"★역작동"
    print(f"{name:8s}  {r_bh:18.3f}  {r_ee:18.3f}  {drop:+7.3f}  {tag}")
print("\n→ 전 변형 낙폭 양수 — 전략−ee가 공통 주식베타를 더 잘 제거함.")
print("  단, 0.49~0.72 잔류는 정상 — 전략 간 공유 timing 구조(위기 시 동시 de-risk)에서 기인.")


## 3. 세 상관행렬 산출

In [ ]:
df_sig=pd.DataFrame({n:signals_common[n].reindex(common_idx) for n in NAMES})
df_act=pd.DataFrame({n:(returns_strat[n].reindex(common_idx)-returns_ee[n].reindex(common_idx)) for n in NAMES})
df_ret=pd.DataFrame({n:returns_strat[n].reindex(common_idx) for n in NAMES})

corr_sig=df_sig.corr(); corr_act=df_act.corr(); corr_ret=df_ret.corr()

# 저장
corr_sig.round(4).to_csv('results/independence_signal_corr.csv')
corr_act.round(4).to_csv('results/independence_active_corr.csv')
corr_ret.round(4).to_csv('results/independence_return_corr.csv')
# 구 정의(전략−B&H)는 이미 independence_active_bh_corr.csv로 보존됨

def show_corr(label, corr):
    print(f"\n{'='*60}"); print(f"  {label}"); print(f"{'='*60}")
    print(f"{'':10s}"+"".join(f"{n:>9s}" for n in NAMES)); print("-"*60)
    for r in NAMES:
        print(f"{r:10s}"+"".join(f"{corr.loc[r,c]:9.3f}" for c in NAMES))

show_corr("(i) 신호 w_t 상관 — 타이밍 겹침 직접 측정", corr_sig)
show_corr("(A) 전략−ee 능동수익 상관 — Gate 1 판정 기준", corr_act)
show_corr("(B) 전략−B&H 능동수익 상관 — 구 정의 (참고)", corr_ret)


## 4. Family 내부 sanity + 내/간 구조

In [ ]:
def classify(r):
    a=abs(r)
    if a<CORR_LOW: return "LOW"
    if a>CORR_HIGH: return "HIGH"
    return "gray"

within_vals=[float(corr_act.loc[a,b]) for a in NAMES for b in NAMES if b>a and FAMILIES[a]==FAMILIES[b]]
between_vals=[float(corr_act.loc[a,b]) for a in NAMES for b in NAMES if b>a and FAMILIES[a]!=FAMILIES[b]]

print(f"family 내부 평균: {sum(within_vals)/len(within_vals):.3f}  "
      f"family 간 평균: {sum(between_vals)/len(between_vals):.3f}")
print("→ 내부 > 간 sanity 통과" if sum(within_vals)/len(within_vals) > sum(between_vals)/len(between_vals) else "★ 역전")

print("\nfamily 간 쌍 (전략−ee 상관 + 신호 w_t 보조):")
for a in NAMES:
    for b in NAMES:
        if b<=a or FAMILIES[a]==FAMILIES[b]: continue
        ra=float(corr_act.loc[a,b]); rs=float(corr_sig.loc[a,b])
        print(f"  {a:8s}↔{b:8s}: ρ_act={ra:+.3f}[{classify(ra)}]  ρ_sig={rs:+.3f}[{classify(rs)}]")


## 5. 신용 행 교차검증: (i) vs (A)

In [ ]:
print(f"{'상대방':8s}  {'(i) 신호 w_t':>14} 분류(i)  {'(A) 전략−ee':>13} 분류(A) 일치")
print("-"*68)
credit_act_cross=[]
for other in [n for n in NAMES if n!='credit']:
    if FAMILIES[other]=='credit': continue
    rs=float(corr_sig.loc['credit',other]); ra=float(corr_act.loc['credit',other])
    credit_act_cross.append(ra)
    match="✓" if classify(rs)==classify(ra) else "다름"
    print(f"{other:8s}  {rs:+14.3f} {classify(rs):>7}  {ra:+13.3f} {classify(ra):>7} {match}")
print(f"\n신용 (A) 상관 범위: {min(credit_act_cross):.3f} ~ {max(credit_act_cross):.3f}")
print(f"모두 HIGH(>{CORR_HIGH}): {all(abs(r)>CORR_HIGH for r in credit_act_cross)}")
print("\n주: (i) 신호 w_t에서 credit↔tsmom12=0.288(LOW)이지만 (A) 전략−ee에서는 0.555(HIGH).")
print("    타이밍이 달라도 위기 시 동시 de-risk로 능동수익 공유 → 두 지표가 다른 결론 제시.")
print(f"    Gate 1은 사전 등록 임계 (A) 기준 적용 — 변호 없음.")


## 6. Gate 1 최종 판정

In [ ]:
crit1={}
print("=== 기준1: 공통기간 Sharpe/Sortino/Calmar vs ee (3개 중 2개 이상) ===")
for name in NAMES:
    ms=common_meta[name]['m_strat']; me=common_meta[name]['m_ee']
    beats=sum([ms['sharpe']>me['sharpe'],ms['sortino']>me['sortino'],ms['calmar']>me['calmar']])
    crit1[name]=(beats>=2)
    print(f"  {name:8s}: Sh {ms['sharpe']:.3f}>{me['sharpe']:.3f}? {ms['sharpe']>me['sharpe']}  "
          f"So {ms['sortino']:.3f}>{me['sortino']:.3f}? {ms['sortino']>me['sortino']}  "
          f"Ca {ms['calmar']:.3f}>{me['calmar']:.3f}? {ms['calmar']>me['calmar']}  → {'PASS' if crit1[name] else 'FAIL'}")

print()
credit_all_high=all(abs(r)>CORR_HIGH for r in credit_act_cross)
credit_any_low =any(abs(r)<CORR_LOW  for r in credit_act_cross)

SOURCE_VARIANTS={'volatility':['vix','realized','blend'],'credit':['credit'],'trend':['ma200','tsmom12']}
include_list=[]
print("="*65)
print("★ Gate 1 최종 판정")
print("="*65)
for src,variants in SOURCE_VARIANTS.items():
    c1=any(crit1[v] for v in variants)
    if src=='credit':
        if c1:         verdict="INCLUDE (기준1)"; include_list.append(src); rsn=f"Sh Δ={common_meta['credit']['m_strat']['sharpe']-common_meta['credit']['m_ee']['sharpe']:+.3f}"
        elif credit_any_low: verdict="INCLUDE (기준2: 저상관)"; include_list.append(src); rsn="저상관 쌍 존재"
        elif not credit_all_high: verdict="INCLUDE (기준2: 회색)"; include_list.append(src); rsn="일부 회색지대"
        else:
            verdict="★ EXCLUDE"
            rsn=(f"기준1 FAIL (Sh {common_meta['credit']['m_strat']['sharpe']:.3f}<ee {common_meta['credit']['m_ee']['sharpe']:.3f}) "
                 f"+ 기준2 FAIL (전략−ee 상관 {min(abs(r) for r in credit_act_cross):.3f}~{max(abs(r) for r in credit_act_cross):.3f}, 모두>{CORR_HIGH})")
    else:
        best=max(variants,key=lambda v:common_meta[v]['m_strat']['sharpe'])
        if c1: verdict="INCLUDE (기준1)"; include_list.append(src); rsn=f"{best} Sh {common_meta[best]['m_strat']['sharpe']:.3f}>ee {common_meta[best]['m_ee']['sharpe']:.3f}"
        else:  verdict="EXCLUDE"; rsn="전 변형 ee 미초과"
    print(f"\n  [{src.upper():12s}]  {verdict}")
    print(f"   근거: {rsn}")

print()
print("─"*65)
print(f"M5 포함 명단: {' + '.join(s.upper() for s in include_list)}")
if 'credit' not in include_list:
    print("CREDIT: 기준1 Sharpe 역전 + 전략−ee 상관 0.555~0.746 (모두>0.50) → EXCLUDE 확정")
print("─"*65)


## 7. 상관 히트맵 저장

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
corr_bh = pd.read_csv('results/independence_active_bh_corr.csv', index_col=0)
panels = [
    (corr_sig,  '(i) 신호 w_t 상관'),
    (corr_bh,   '(B) 전략−B&H 구 정의 (공통베타 잔류)'),
    (corr_act,  '(A) 전략−ee 새 정의 (Gate 1 기준)'),
]
for ax,(corr,title) in zip(axes,panels):
    im=ax.imshow(corr.values,vmin=-1,vmax=1,cmap='RdYlGn',aspect='auto')
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels(NAMES,rotation=45,ha='right',fontsize=8)
    ax.set_yticklabels(NAMES,fontsize=8)
    for i in range(6):
        for j in range(6):
            v=float(corr.iloc[i,j])
            ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=7,color='black' if abs(v)<0.7 else 'white')
    ax.set_title(title,fontsize=8,fontweight='bold')
    plt.colorbar(im,ax=ax,shrink=0.8)
plt.suptitle(f'M4 독립성(수정) — {COMMON_START.date()}~  Gate 1: |ρ_active-ee|<{CORR_LOW}(저)/>{CORR_HIGH}(고)',fontsize=9,y=1.01)
plt.tight_layout()
plt.savefig('results/independence_corr_heatmap.png',dpi=120,bbox_inches='tight')
plt.close()
print("저장: independence_corr_heatmap.png")


## 8. Gate 1 해석 노트

### 능동수익 정의 변경 효과
| 정의 | 대표 범위 (family 간) | 이유 |
|---|---|---|
| 전략 − B&H | 0.70 ~ 0.89 (모두 HIGH) | 공통 주식베타 `(1-w)×r_eq` 잔류 |
| **전략 − ee** | **0.56 ~ 0.82 (HIGH 유지)** | 베타 일부 제거, timing 구조 잔류 |

전략−ee로도 HIGH 대역이 유지되는 이유: 세 정보원이 모두 "위기 시 노출 축소" 전략이라
ee 대비 초과수익(timing alpha)이 위기 이벤트(2008·2020·2022)에서 동시 발생 → 상관.

### 신용 교차검증 해석
- 신호 w_t 상관(i)에서 credit↔tsmom12=0.288(LOW) — 타이밍이 다름.
- 전략−ee 상관(A)에서 credit↔tsmom12=0.555(HIGH) — 위기 시 능동수익은 공유.
- 두 지표가 다른 결론: Gate 1은 사전 등록 임계(A) 적용 → **CREDIT EXCLUDE 확정**.
- 정직한 음의 결론: BAA10Y 단독 타이밍이 ee를 이기지 못하고, ee 차감 후도 HIGH 상관.

### M5 진입: VOLATILITY + TREND
- vol↔trend 전략−ee 상관 0.695~0.819도 HIGH.
- M5 결합이 단순 노출 축소 대비 분산이득을 제한적으로만 제공할 수 있음을 인식.
- **Gate 2(M5)가 최종 판정 — 실제 표본외 결합 성과로 확인한다.**
